# Introduction to Word Embeddings

This notebook introduces word embeddings -- dense vector representations of words that capture semantic meaning.

We will:
1. Explain the concept with diagrams.
2. Use TruncatedSVD on a TF-IDF matrix to create simple word vectors.
3. Visualize word relationships with 2D PCA scatter plots.
4. Compute cosine similarity between words.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

sns.set_style('whitegrid')
print('Imports ready.')

## 1. What Are Word Embeddings?

Traditional representations (one-hot encoding, bag-of-words) treat words as independent symbols. Word embeddings map words to dense, low-dimensional vectors where **similar words are close together** in the vector space.

In [ ]:
# Conceptual diagram: One-Hot vs Dense Embeddings
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# One-hot encoding
words = ['king', 'queen', 'man', 'woman', 'cat']
one_hot = np.eye(5)
ax1.imshow(one_hot, cmap='Blues', aspect='auto')
ax1.set_xticks(range(5))
ax1.set_xticklabels([f'dim{i}' for i in range(5)], rotation=45)
ax1.set_yticks(range(5))
ax1.set_yticklabels(words)
ax1.set_title('One-Hot Encoding\n(Sparse, High-Dimensional)', fontsize=12)
for i in range(5):
    for j in range(5):
        ax1.text(j, i, f'{one_hot[i,j]:.0f}', ha='center', va='center', fontsize=11)

# Dense embeddings (simulated)
embeddings = np.array([
    [0.8, 0.6, 0.9],   # king
    [0.7, 0.7, 0.8],   # queen
    [0.6, 0.3, 0.5],   # man
    [0.5, 0.4, 0.4],   # woman
    [0.1, 0.9, 0.1],   # cat
])
ax2.imshow(embeddings, cmap='Oranges', aspect='auto')
ax2.set_xticks(range(3))
ax2.set_xticklabels(['royalty', 'animate', 'power'], rotation=45)
ax2.set_yticks(range(5))
ax2.set_yticklabels(words)
ax2.set_title('Dense Embeddings\n(Semantic Dimensions)', fontsize=12)
for i in range(5):
    for j in range(3):
        ax2.text(j, i, f'{embeddings[i,j]:.1f}', ha='center', va='center', fontsize=11)

plt.suptitle('One-Hot Encoding vs Word Embeddings', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Diagram: Word embedding space (conceptual 2D illustration)
fig, ax = plt.subplots(figsize=(10, 8))

concept_words = {
    'Animals':   {'cat': (1, 2), 'dog': (1.5, 2.3), 'fish': (0.8, 1.5), 'bird': (1.2, 2.8)},
    'Royalty':   {'king': (5, 6), 'queen': (5.5, 6.3), 'prince': (4.8, 5.5), 'princess': (5.3, 5.8)},
    'Countries': {'france': (7, 2), 'germany': (7.5, 2.5), 'spain': (6.5, 1.8), 'italy': (7.2, 1.5)},
    'Tech':      {'computer': (3, 8), 'software': (3.5, 8.3), 'algorithm': (2.8, 7.5), 'data': (3.3, 7.8)},
}

colors_map = {'Animals': '#e74c3c', 'Royalty': '#9b59b6', 'Countries': '#2ecc71', 'Tech': '#3498db'}

for category, words_dict in concept_words.items():
    for word, (x, y) in words_dict.items():
        ax.scatter(x, y, c=colors_map[category], s=100, zorder=3)
        ax.annotate(word, (x, y), fontsize=11, fontweight='bold',
                   xytext=(5, 5), textcoords='offset points')

handles = [mpatches.Patch(color=c, label=cat) for cat, c in colors_map.items()]
ax.legend(handles=handles, fontsize=11, loc='center left')
ax.set_title('Conceptual Word Embedding Space\n(Similar words cluster together)', fontsize=14)
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
plt.tight_layout()
plt.show()

## 2. Building Word Vectors with TruncatedSVD on TF-IDF

One of the simplest ways to create word embeddings is to apply dimensionality reduction (SVD) to a term-document TF-IDF matrix. This is the idea behind Latent Semantic Analysis (LSA).

In [ ]:
# Load corpus
categories = ['sci.space', 'rec.sport.baseball', 'comp.graphics', 'talk.politics.guns']
newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'))

print(f'Loaded {len(newsgroups.data)} documents.')

In [ ]:
# Build TF-IDF matrix (documents x words)
tfidf = TfidfVectorizer(max_features=5000, stop_words='english',
                         max_df=0.9, min_df=5)
tfidf_matrix = tfidf.fit_transform(newsgroups.data)
vocab = tfidf.get_feature_names_out()

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Vocabulary size: {len(vocab)}')

In [ ]:
# Apply TruncatedSVD to get word vectors
# We transpose the matrix so that rows=words, columns=documents,
# then reduce to n_components dimensions.
n_components = 50
svd = TruncatedSVD(n_components=n_components, random_state=42)

# Apply SVD on the transposed matrix to get word embeddings
word_vectors = svd.fit_transform(tfidf_matrix.T)
word_vectors = normalize(word_vectors)  # L2 normalize

print(f'Word vectors shape: {word_vectors.shape}')
print(f'Explained variance ratio (first 10): {svd.explained_variance_ratio_[:10].round(4)}')
print(f'Total explained variance: {svd.explained_variance_ratio_.sum():.4f}')

## 3. Cosine Similarity Between Words

In [ ]:
def get_word_vector(word):
    """Get the embedding vector for a word."""
    idx = np.where(vocab == word)[0]
    if len(idx) == 0:
        return None
    return word_vectors[idx[0]]

def word_similarity(word1, word2):
    """Compute cosine similarity between two words."""
    v1, v2 = get_word_vector(word1), get_word_vector(word2)
    if v1 is None or v2 is None:
        return None
    return cosine_similarity(v1.reshape(1, -1), v2.reshape(1, -1))[0, 0]

def most_similar(word, top_n=10):
    """Find the most similar words to a given word."""
    v = get_word_vector(word)
    if v is None:
        return []
    sims = cosine_similarity(v.reshape(1, -1), word_vectors)[0]
    top_idx = np.argsort(sims)[::-1][1:top_n+1]  # Skip the word itself
    return [(vocab[i], sims[i]) for i in top_idx]

# Test similarity
test_pairs = [
    ('space', 'orbit'), ('space', 'nasa'), ('space', 'baseball'),
    ('gun', 'weapon'), ('gun', 'graphics'), ('game', 'team'),
    ('image', 'graphics'), ('image', 'baseball'),
]

print('Word Pair Cosine Similarities:')
print('-' * 45)
for w1, w2 in test_pairs:
    sim = word_similarity(w1, w2)
    if sim is not None:
        bar = '#' * int(sim * 30)
        print(f'  {w1:12s} - {w2:12s}: {sim:+.4f}  {bar}')

In [ ]:
# Most similar words
query_words = ['space', 'baseball', 'gun', 'image']

for word in query_words:
    similar = most_similar(word, top_n=8)
    if similar:
        print(f'\nMost similar to "{word}":')
        for w, s in similar:
            print(f'  {w:15s}  {s:.4f}')

## 4. Visualize Word Embeddings in 2D

We use PCA to project the word vectors down to 2D for visualization.

In [ ]:
# Select interesting words to visualize
words_to_plot = {
    'Space':    ['space', 'orbit', 'nasa', 'launch', 'shuttle', 'moon', 'earth', 'satellite'],
    'Baseball': ['baseball', 'game', 'team', 'player', 'hit', 'score', 'season', 'pitch'],
    'Graphics': ['image', 'graphics', 'display', 'color', 'screen', 'pixel', 'render', 'software'],
    'Guns':     ['gun', 'weapon', 'fire', 'law', 'police', 'crime', 'rights', 'control'],
}

# Gather vectors and labels
plot_words = []
plot_vectors = []
plot_categories = []

for category, word_list in words_to_plot.items():
    for word in word_list:
        v = get_word_vector(word)
        if v is not None:
            plot_words.append(word)
            plot_vectors.append(v)
            plot_categories.append(category)

plot_vectors = np.array(plot_vectors)
print(f'Plotting {len(plot_words)} words.')

In [ ]:
# PCA to 2D
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(plot_vectors)

fig, ax = plt.subplots(figsize=(14, 10))

cat_colors = {'Space': '#e74c3c', 'Baseball': '#2ecc71', 'Graphics': '#3498db', 'Guns': '#f39c12'}

for i, (word, cat) in enumerate(zip(plot_words, plot_categories)):
    ax.scatter(coords_2d[i, 0], coords_2d[i, 1], c=cat_colors[cat], s=120, zorder=3, edgecolors='black', linewidths=0.5)
    ax.annotate(word, (coords_2d[i, 0], coords_2d[i, 1]),
               fontsize=11, fontweight='bold',
               xytext=(8, 5), textcoords='offset points',
               color=cat_colors[cat])

handles = [mpatches.Patch(color=c, label=cat) for cat, c in cat_colors.items()]
ax.legend(handles=handles, fontsize=12, loc='best')
ax.set_title('Word Embeddings Projected to 2D (PCA)\nWords from the same topic cluster together', fontsize=14)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Similarity Heatmap

In [ ]:
# Compute pairwise similarity for selected words
selected_words = ['space', 'orbit', 'nasa', 'baseball', 'team', 'game',
                   'image', 'graphics', 'gun', 'weapon', 'law']

selected_vectors = []
valid_words = []
for w in selected_words:
    v = get_word_vector(w)
    if v is not None:
        selected_vectors.append(v)
        valid_words.append(w)

sim_matrix = cosine_similarity(np.array(selected_vectors))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sim_matrix, annot=True, fmt='.2f', cmap='RdYlBu_r',
            xticklabels=valid_words, yticklabels=valid_words, ax=ax,
            vmin=-0.5, vmax=1.0)
ax.set_title('Cosine Similarity Heatmap Between Words', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

Key takeaways:
1. **Word embeddings** represent words as dense vectors where similar words are close together.
2. **TruncatedSVD on TF-IDF** (a.k.a. LSA) is a simple way to create word vectors from a corpus.
3. **Cosine similarity** measures how related two word vectors are.
4. **PCA visualization** confirms that words from the same topic cluster together in the embedding space.

For production use, pre-trained embeddings (Word2Vec, GloVe, FastText) or contextual embeddings (BERT) are typically preferred over LSA.